# Lab 02: Multi-Source Bronze Ingestion & Initial DQ Profiling
**Course:** Business Intelligence and Big Data Analysis (`01 31 0367 00`)  
**Target Audience:** 4th Year B.Sc. Students, Management and Production Engineering  
**Duration:** 3 Hours (180 Minutes)  
**Prerequisite:** Lab 01 (`.venv` setup & Python Primer completed)

---

## 🎯 1. Context & Architectural Objective
In industrial BI pipelines, production data is never delivered in one clean file. Every case study in this course has **3 heterogeneous raw formats**:
- **SmartPress Automotive (Default Reference):** `sensor_telemetry.csv` (SCADA), `maintenance_logs.json` (CMMS), `machines_and_shifts.xlsx` (ERP Master: `Machines`, `Shifts`, `Product_Variants`)
- **DataCo Hydraulics:** `order_items_trans.csv`, `shipment_events.json`, `master_catalog.xlsx`
- **Nexis Workforce:** `shift_production_performance.csv`, `workforce_events.json`, `workforce_roster_and_skills.xlsx`

In this lab, you will act as a **Data Engineer** building the **Bronze Layer**:
1. Dynamically resolve paths across both student and reference repositories.
2. Ingest **CSV**, **JSON**, and **multi-sheet Excel** sources with `pandas`.
3. Inject enterprise audit lineage columns (`_ingestion_timestamp_utc`, `_source_file`).
4. Stage immutable tables to **Snappy-compressed Parquet** inside `sandbox_outputs/<case>/bronze_staged/` (`index=False`).
5. Profile schema, duplicates, casing/whitespace noise, nulls, and physical outliers **without deleting rows** (cleaning happens in **Labs 03–04**).

---

## ⚙️ 2. Environment Check & Dynamic Path Discovery
Verify `.venv` selection (`pandas`, `openpyxl`, `pyarrow`) and locate `BRONZE_RAW` without editing raw files.

### Step 1: Verify Libraries & Locate Raw Files

In [ ]:
from pathlib import Path

_cwd = Path.cwd().resolve()
REPO_ROOT = None
for _cand in [_cwd, *_cwd.parents]:
    if (_cand / "03_labs_student_starters").exists():
        REPO_ROOT = _cand
        break
    for _sub in ("ife_bi_bigdata_course", "ife_bi_bigdata_student"):
        if (_cand / _sub / "03_labs_student_starters").exists():
            REPO_ROOT = _cand / _sub
            break
    if REPO_ROOT is not None:
        break
assert REPO_ROOT is not None, f"Could not locate 03_labs_student_starters from {_cwd}"

CASE_NAME = "case1_smartpress_automotive"
STUDENT_BRONZE_DIR = REPO_ROOT / "03_labs_student_starters" / "lab_02_bronze_ingestion_and_profiling" / "data" / CASE_NAME
REF_CASE_DATA = REPO_ROOT / "05_case_studies_reference_solutions" / CASE_NAME / "data"
BRONZE_RAW = STUDENT_BRONZE_DIR if STUDENT_BRONZE_DIR.exists() else (REF_CASE_DATA / "bronze")
SANDBOX_DATA = REPO_ROOT / "03_labs_student_starters" / "sandbox_outputs" / CASE_NAME


def resolve_stage_dir(stage_name: str) -> Path:
    """Read upstream stage from student sandbox_outputs if populated, else fallback read-only to reference solution."""
    cand = SANDBOX_DATA / stage_name
    if cand.exists() and any(cand.glob("*.parquet")):
        return cand
    return REF_CASE_DATA / stage_name
import json
from datetime import datetime, timezone
import pandas as pd

BRONZE_STAGED = SANDBOX_DATA / "bronze_staged"
BRONZE_STAGED.mkdir(parents=True, exist_ok=True)

print("Repository Root :", REPO_ROOT)
print("Bronze Raw Dir  :", BRONZE_RAW, "| Exists:", BRONZE_RAW.exists())
print("Bronze Staged   :", BRONZE_STAGED)

assert BRONZE_RAW.exists(), f"Missing raw data directory: {BRONZE_RAW}"
for p in sorted(BRONZE_RAW.iterdir()):
    print(f"  - {p.name:<32} ({p.stat().st_size / 1024:,.1f} KB)")


## 📥 3. Part 1: Ingest Telemetry CSV & Attach Audit Metadata
Load `sensor_telemetry.csv`, add `_ingestion_timestamp_utc` and `_source_file`, then inspect `.shape` and `.info()`.

In [ ]:
csv_name = "sensor_telemetry.csv"
now_utc = datetime.now(timezone.utc).isoformat()

# TODO: Load CSV using pd.read_csv(BRONZE_RAW / csv_name)
df_telemetry = None  # REPLACE WITH YOUR CODE

# TODO: Attach _ingestion_timestamp_utc = now_utc and _source_file = csv_name

if df_telemetry is not None:
    print("Telemetry Shape:", df_telemetry.shape)
    print(df_telemetry.head(3).to_string())


## 📋 4. Part 2: Ingest Semi-Structured JSON Maintenance Logs
Parse `maintenance_logs.json` with `json.load()` and flatten it into `df_maintenance` with audit metadata.

In [ ]:
json_name = "maintenance_logs.json"

# TODO: Open BRONZE_RAW / json_name, load JSON records, convert to pd.DataFrame, and attach audit columns
df_maintenance = None  # REPLACE WITH YOUR CODE

if df_maintenance is not None:
    print("Maintenance Shape:", df_maintenance.shape)
    print(df_maintenance.head(3).to_string())


## 📊 5. Part 3: Ingest Multi-Sheet Excel Master Data (`Machines`, `Shifts`, `Product_Variants`)
Inspect `machines_and_shifts.xlsx` sheet names and load each tab as its own entity DataFrame.

In [ ]:
excel_name = "machines_and_shifts.xlsx"
excel_path = BRONZE_RAW / excel_name

# TODO: Inspect sheets with pd.ExcelFile(excel_path).sheet_names and read all 3 tabs
df_machines = None          # REPLACE WITH YOUR CODE
df_shifts = None            # REPLACE WITH YOUR CODE
df_product_variants = None  # REPLACE WITH YOUR CODE

# TODO: Attach _ingestion_timestamp_utc and _source_file to all 3 DataFrames


## 💾 6. Part 4: Stage All Tables to Snappy Parquet (`index=False`)
Persist all 5 raw tables verbatim to `BRONZE_STAGED/*.parquet` without writing Pandas integer index columns.

In [ ]:
# TODO: Export all 5 DataFrames to BRONZE_STAGED with index=False and compression="snappy"
outputs = {
    "bronze_telemetry.parquet": df_telemetry,
    "bronze_maintenance.parquet": df_maintenance,
    "bronze_machines.parquet": df_machines,
    "bronze_shifts.parquet": df_shifts,
    "bronze_product_variants.parquet": df_product_variants,
}

# WRITE YOUR EXPORT LOOP HERE


## 🔍 7. Initial Data Quality Profiling (Read-Only Diagnosis)
Reload `bronze_telemetry.parquet` and `bronze_maintenance.parquet` and catalog every anomaly **without modifying rows**. We will quarantine and clean them in **Labs 03–04**.

In [ ]:
# TODO: Inspect unique machine_ids, physical torque bounds, NaN counts, duplicates, and CMMS downtime labels
